# TREAD-ONE NB05 — one paired shared-gradient comparison

**This notebook launches the next measured study after automatic preflight:**
one unchanged TREAD-ONE architecture, **seed 1**, two matched conditions,
**60 epochs / 2,940 optimizer updates each**. Baseline uses the original summed
gradient; `shared_projection` applies symmetric conflict projection only to
shared modules. Initial weights, samples, augmentation, losses, optimizer,
validation selection and endpoints are matched. It is one unreplicated paired
development pilot, not established generalization or an improved model claim.

## Kaggle setup

1. **T4 ×2**, Internet **ON**; attach exactly one expanded
   `shanmuk4622/tire-dataset-prepared-phase2` dataset.
2. Enable write-authorized Kaggle Secret **HF_TOKEN** for
   `Shanmuk4622/tyre-wear-study`.
3. Use a clean working area with **12 GiB free**. The prepared dataset stays in
   input; no pretrained weights or old training ZIP is downloaded.
4. **Run All**. No dependency downgrade or new package installation.

**29 October lease-wait repair:** if another session holds the automatic
preflight's HF writer lease, this notebook waits and safely retries every five
minutes, for up to 90 minutes of waiting. Stop cancels waiting. An abandoned
lease expires within 45 minutes of its last renewal; an active session can
renew it. Stop an unused older session cleanly. Do not delete OWNER.json or
change the experiment namespace. Training errors are not automatically retried.
Attempt logs remain in `/kaggle/working/treadone_r2_paired/launcher_logs/`.
The 27 experiment sources and checkpoint/source contract are unchanged.

The first short automatic preflight exercises both conditions with validation
fixtures, native/CSV recovery and actual pinned HF download. It must pass before
the two measured runs start. The old NB04 and r1 data/checkpoints are preserved.

## Retention and resume

Every actual augmented training occurrence, loss sufficient statistic, pixel
and point error, latent/evidence statistic and task-gradient observation is
retained. Full validation runs over all 81 images each epoch. Selected and final
endpoints retain predictions and native maps for all 103 previously inspected
development test images. Source/test limitations remain explicit.

Local atomic journal units keep native optimizer/scheduler/AMP/RNG state and
observation cursors together. Every published metric table is a **direct CSV**;
native PT/NPY and JSON supplement it. No HF ZIP/NPZ or SQLite upload.

HF publication is batched every **30 minutes**, after major stages, or on a
catchable Stop/error. Immutable direct files are staged in bounded batches; the
snapshot manifest selects only verified objects. Forced kernel death cannot
flush unsent progress. Native local states rotate to two; all scalar records,
declared bounded maps and selected-weight improvements remain retained.

Rerun this notebook unchanged to restore compatible published state. A changed
software/source contract is rejected rather than silently starting over.
Keep the session open while Stop flushes and server backoff is active. Do not
launch another writer for the same condition.

The model outputs remain classification, overlapping tyre/tread masks,
mask-derived boxes, ordered points and exact additive signed evidence. No app
model is replaced automatically. More independent tyre data remains needed.


In [1]:
import base64,gzip,json,os,signal,subprocess,sys,time
from pathlib import Path
from kaggle_secrets import UserSecretsClient
from filelock import FileLock
WORK=Path('/kaggle/working/treadone_r2_paired')
WORK.mkdir(parents=True,exist_ok=True)
os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN')
sources=json.loads(gzip.decompress(base64.b64decode('...')))
with FileLock(str(WORK/'R2_SESSION.lock'),timeout=1):
    for name,value in sources.items():
        assert not Path(name).is_absolute() and '..' not in Path(name).parts
        target=WORK/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(base64.b64decode(value))
print('TREAD-ONE NB05 ready; secret loaded without displaying it.')

"""Notebook-only lease wait; the 27 frozen experiment sources stay unchanged."""
import json
import os
import signal
import subprocess
import sys
import time
import uuid
from pathlib import Path
from filelock import FileLock

LEASE_ERROR = ('RuntimeError: Another live gate owns this namespace; stop it '
               'cleanly or wait for its 45-minute lease expiry.')


def preflight_lease_busy(text):
    """Retry only the observed preflight claim failure, never a training error."""
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return bool(lines and lines[-1] == LEASE_ERROR
                and 'in preflight' in text and 'in claim' in text
                and 'Measured paired condition:' not in text)


def _worker_progress(work, seen):
    for path in (work / 'revision2_runs').glob('**/console.log'):
        # Logs can grow during a long run; read only their tail.
        with open(path, 'rb') as stream:
            stream.seek(max(0, path.stat().st_size - 8192))
            lines = stream.read().decode(errors='replace').splitlines()
        if not lines or seen.get(str(path)) == lines[-1]:
            continue
        seen[str(path)] = lines[-1]
        label = (path.parent.name if path.parent.name in ('baseline', 'shared_projection')
                 else path.parent.parent.name + '/' + path.parent.name)
        print(label + ': ' + lines[-1], flush=True)


def _stop_child(child):
    if child.poll() is not None:
        return
    print('Stop requested: preserving committed units and attempting HF flush. '
          'Keep the session open.', flush=True)
    try:
        os.killpg(child.pid, signal.SIGINT)
    except ProcessLookupError:
        return
    for index in range(60):
        try:
            child.wait(timeout=5)
            return
        except subprocess.TimeoutExpired:
            if index % 4 == 0:
                print('Still flushing or respecting server delay; do not launch '
                      'a duplicate writer.', flush=True)
    print('Coordinator is still flushing; forced termination can lose unsent work.',
          flush=True)


def launch_paired(work, retry_seconds=300, max_lease_wait_seconds=5400):
    """Recheck occupied preflight leases every five minutes, for up to 90 minutes.

    Each retry calls the unchanged coordinator, which performs its normal CAS
    claim and compatibility checks. This launcher never edits remote ownership.
    A live writer that renews its lease remains protected. Stop cancels waiting.
    """
    assert retry_seconds > 0 and max_lease_wait_seconds >= 0
    work = Path(work)
    logs = work / 'launcher_logs'
    logs.mkdir(parents=True, exist_ok=True)
    run_id = time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()) + '_' + uuid.uuid4().hex[:8]
    history = logs / (run_id + '.jsonl')

    def record(event, **values):
        with open(history, 'a', encoding='utf-8') as stream:
            stream.write(json.dumps(dict(event=event, time_utc=time.strftime(
                '%Y-%m-%dT%H:%M:%SZ', time.gmtime()), **values)) + '\n')
            stream.flush()
            os.fsync(stream.fileno())

    waited = 0.0
    attempt = 0
    child = None
    seen = {}
    # Includes the wait interval; a second launch cell cannot start another waiter.
    with FileLock(str(work / 'NB05_LAUNCHER.lock'), timeout=1):
        try:
            while True:
                attempt += 1
                log = logs / (run_id + f'_attempt_{attempt:02d}.log')
                record('coordinator_start', attempt=attempt, log=str(log))
                with open(log, 'wb', buffering=0) as output, open(log, 'rb') as reader:
                    child = subprocess.Popen(
                        [sys.executable, '-B', '-u', str(work / 'revision2/coordinator.py'),
                         '--publish'], stdout=output, stderr=subprocess.STDOUT,
                        start_new_session=True)
                    while True:
                        try:
                            code = child.wait(timeout=5)
                        except subprocess.TimeoutExpired:
                            code = None
                        while True:
                            chunk = reader.read(65536)
                            if not chunk:
                                break
                            print(chunk.decode(errors='replace'), end='', flush=True)
                        _worker_progress(work, seen)
                        if code is not None:
                            break
                record('coordinator_exit', attempt=attempt, exit_code=code)
                if code == 0:
                    record('completed', lease_wait_seconds=waited)
                    return
                with open(log, 'rb') as stream:
                    stream.seek(max(0, log.stat().st_size - 32768))
                    tail = stream.read().decode(errors='replace')
                if not preflight_lease_busy(tail):
                    raise RuntimeError('NB05 stopped. Inspect the error above and ' + str(log)
                                       + '. Committed progress is preserved; this error '
                                       'is not retried automatically.')
                remaining = max_lease_wait_seconds - waited
                if remaining <= 0:
                    record('lease_wait_exhausted', lease_wait_seconds=waited)
                    raise RuntimeError('The preflight namespace is still occupied after '
                                       f'{waited / 60:g} minutes of waiting. Stop the other '
                                       'session cleanly, or let its lease expire, then rerun '
                                       'this notebook. No owner or checkpoint was overwritten.')
                delay = min(float(retry_seconds), remaining)
                record('preflight_lease_wait', attempt=attempt, seconds=delay,
                       cumulative_wait_seconds=waited)
                print('HF preflight lease is occupied. Waiting '
                      f'{delay / 60:g} minutes before a safe retry '
                      f'({waited / 60:g} of {max_lease_wait_seconds / 60:g} minutes waited). '
                      'An abandoned lease expires within 45 minutes of its last renewal; '
                      'an active session may renew it. Stop cancels this wait. '
                      'No lock is deleted and no training is started while occupied.', flush=True)
                deadline = time.monotonic() + delay
                while time.monotonic() < deadline:
                    time.sleep(min(5.0, max(0.0, deadline - time.monotonic())))
                waited += delay
        except KeyboardInterrupt:
            record('user_stop', attempt=attempt, lease_wait_seconds=waited)
            if child is not None:
                _stop_child(child)
            raise


def launch():
    return launch_paired(WORK)


TREAD-ONE NB05 ready; secret loaded without displaying it.


## Automatic live preflight, then the paired 60-epoch runs
Both conditions use the same frozen architecture. This cell starts the declared measured study only after preflight passes.

In [2]:
launch()

Automatic live preflight: baseline
preflight_baseline/remote_preflight: baseline updates 1 epoch 1 phase train
preflight_baseline/expected: baseline updates 2 epoch 1 phase validation
preflight_baseline/expected: baseline updates 4 epoch 2 phase validation
preflight_baseline/paused: baseline updates 1 epoch 1 phase train
snapshots/unit_00000001-61be540b: baseline updates 1 epoch 1 phase train
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...0000001-61be540b/state.pt:  99%|█████████▉| 16.4MB / 16.6MB            

Processing Files (0 / 1)      :  99%|█████████▉| 16.4MB / 16.6MB, 1.59MB/s  

Processing Files (1 / 1)      : 100%|██████████| 16.6MB / 16.6MB, 1.59MB/s  
New Data Upload               : 100%|██████████|  194kB /  194kB, 18.7kB/s  

  ...0000001-61be540b/state.pt: 100%|██████████| 16.6MB / 16.6MB            

  ...0000001-61be540b/state.pt: 100%|██████████| 16.6MB / 16.6MB    

## Results and interactive progress reports
Interpret this as a one-seed development comparison. Installation requires separate review. No test-based checkpoint or seed reselection.

In [3]:
from IPython.display import display,FileLink
latest=json.loads((WORK/'revision2_runs/LATEST_PAIR.json').read_text())
folder=Path(latest['path'])
result=json.loads((folder/'PAIR_DELIVERY.json').read_text())
assert result['status']=='completed'
print(json.dumps(result,indent=2))
for condition in ('baseline','shared_projection'):
    snapshot=json.loads((folder/condition/'LATEST_SNAPSHOT.json').read_text())['path']
    display(FileLink(str(Path(snapshot)/'TREAD_ONE_R2_Progress.html')))
    display(FileLink(str(Path(snapshot)/'artifacts/PAIR_COMPARISON.csv')))
print('Preserve this executed notebook for independent publication/result audit.')


{
  "epochs_per_run": 60,
  "measured_runs": 2,
  "optimizer_updates_per_run": 2940,
  "preflight": [
    {
      "actual_live_HF_restore": true,
      "completed_updates": 4,
      "condition": "baseline",
      "measured_study_started": false,
      "native_differences": {
        "best": 0,
        "cursor": 0,
        "epoch0": 0,
        "eval_cursor": 0,
        "history": 0,
        "losses": 0.0,
        "model": 0.0,
        "optimizer": 0.0,
        "partial": 0,
        "phase": 0.0,
        "rng": 0,
        "row_cursors": 0,
        "scaler": 0.0,
        "scheduler": 0,
        "updates": 0
      },
      "original_r1_update_math": {
        "model": 0.0,
        "optimizer": 0.0,
        "rng": 0,
        "scaler": 0.0,
        "scheduler": 0
      },
      "protocol": "e78b44d3b077c1999635b855f57bf1fd90b6d6fd72370a1fd63dfd6a8b9c0e97",
      "published_source_revision": "e136f578602bf522614a00aea176c98fa22f450e",
      "restored_updates": 1,
      "scientific_tables_equa

/kaggle/working/treadone_r2_paired/revision2_runs/16dfddd7672e70c3b321cc6adc4044d5a6c4453a75d7b84a29f72fdd6ab6c1dc/baseline/snapshots/unit_00003627-909f248c/TREAD_ONE_R2_Progress.html

/kaggle/working/treadone_r2_paired/revision2_runs/16dfddd7672e70c3b321cc6adc4044d5a6c4453a75d7b84a29f72fdd6ab6c1dc/baseline/snapshots/unit_00003627-909f248c/artifacts/PAIR_COMPARISON.csv

/kaggle/working/treadone_r2_paired/revision2_runs/16dfddd7672e70c3b321cc6adc4044d5a6c4453a75d7b84a29f72fdd6ab6c1dc/shared_projection/snapshots/unit_00003627-2224da46/TREAD_ONE_R2_Progress.html

/kaggle/working/treadone_r2_paired/revision2_runs/16dfddd7672e70c3b321cc6adc4044d5a6c4453a75d7b84a29f72fdd6ab6c1dc/shared_projection/snapshots/unit_00003627-2224da46/artifacts/PAIR_COMPARISON.csv

Preserve this executed notebook for independent publication/result audit.
